In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dateutil import tz

import datetime
import math

In [ ]:
smesh = pd.read_csv("external_data/pepperwood2025/concatSmeshData/f864_airQualityMetrics.csv")
smesh.sort_values(by = "datetime", inplace = True, ignore_index = True)
smesh['datetime'] = pd.to_datetime(smesh['datetime'])
smesh['datetime'] = smesh['datetime'].dt.tz_localize("US/Pacific", ambiguous=True)


smesh.set_index("datetime", inplace = True)
smesh = smesh["pm25Environmental"].resample("10min").mean()

display(smesh.head)
# smesh["datetime"] = smesh["datetimeCol"].iloc(1)

print(smesh)

In [ ]:
purpleAirSensorLocations = pd.read_csv("PurpleAir Download 11-7-2025/PurpleAirLocations.csv")
display(purpleAirSensorLocations.head())
dataArray = []
for sensorName in purpleAirSensorLocations["Sensor Name"]:
    dataArray.append(pd.read_csv("PurpleAir Download 11-7-2025/" + sensorName + ".csv"))
    
for i, data in enumerate(dataArray):
    data['datetime'] = pd.to_datetime(data['time_stamp'])
    # data = data[data['pm2.5_atm'] <= 10040]
    # data.loc[:,'pm2.5_atm'] = data['pm2.5_atm'].where(data['pm2.5_atm'] <= 80, 0)


    data.set_index("datetime", inplace = True)
    # data.index = data.index.tz_convert("UTC") 
    dataResampled = data[["humidity", "temperature", "pressure", "pm2.5_atm"]].resample('10min').mean()
    dataArray[i] = dataResampled


pepperwoodPreserve = dataArray[purpleAirSensorLocations.index[purpleAirSensorLocations["Sensor Name"] == "Pepperwood Preserve"][0]]
franzValley= dataArray[purpleAirSensorLocations.index[purpleAirSensorLocations["Sensor Name"] == "Franz Valley"][0]]
meadowSprings = dataArray[purpleAirSensorLocations.index[purpleAirSensorLocations["Sensor Name"] == "Meadow Springs"][0]]
outside8566 = dataArray[purpleAirSensorLocations.index[purpleAirSensorLocations["Sensor Name"] == "8566 Outside"][0]]


print(len(pepperwoodPreserve))


In [ ]:
fig, axes = plt.subplots(len(dataArray) + 1, 1, figsize = (8,10))

# ax[0].plot(pepperwoodPreserve["pm2.5_atm"])
# ax[1].plot(smesh["pm25Environmental"])


low = pd.Timestamp("2025-10-23-09", tz = "US/Pacific")
# low = low.tz_localize("US/Pacific")
high = pd.Timestamp("2025-10-23-18", tz = "US/Pacific")
# high = high.tz_localize("US/Pacific")

splicedSmesh = smesh.loc[low:high]
axes[0].plot(splicedSmesh.index, splicedSmesh)
print(len(splicedSmesh))
axes[0].set_title("Smesh Sensor")

axes[0].tick_params(axis='x', rotation=45)

correlationMatrix = pd.DataFrame()
correlationMatrix["Smesh Sensor"] = splicedSmesh.to_list()
for i in range(len(dataArray)):

    splicedData = dataArray[i].loc[low:high]
    axes[i + 1].plot(splicedData.index,splicedData["pm2.5_atm"])
    print(len(splicedData["pm2.5_atm"]))
    correlationMatrix[purpleAirSensorLocations["Sensor Name"][i]] = splicedData["pm2.5_atm"].to_list()

    axes[i + 1].set_title(purpleAirSensorLocations["Sensor Name"][i])
    axes[i+1].tick_params(axis='x', rotation=45)



# high = pd.Timestamp("2025-10-24-12")
# high = pepperwoodPreserve.index.max()

for ax in axes:

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:00',tz=tz.gettz("US/Pacific"))) 

plt.subplots_adjust(hspace=1)


plt.show()

In [ ]:
#Does our smesh data measure air quality
#USe wind and Smesh Data to compare with Purple Air


#Alert Calofornia, started to go north east, then switched to southwest
#Extraplolate the Wind data we had from our working wind data



In [ ]:
fig, axes = plt.subplots(len(dataArray) + 1, 1, figsize = (10,12))

# ax[0].plot(pepperwoodPreserve["pm2.5_atm"])
# ax[1].plot(smesh["pm25Environmental"])


low = pd.Timestamp("2025-10-24-12", tz = "US/Pacific")
# low = low.tz_localize("US/Pacific")
high = pd.Timestamp("2025-10-25-18", tz = "US/Pacific")
# high = high.tz_localize("US/Pacific")

splicedSmesh = smesh.loc[low:high]
axes[0].plot(splicedSmesh.index, splicedSmesh)
print(len(splicedSmesh))
axes[0].set_title("Smesh Sensor")
axes[0].tick_params(axis='x', labelrotation=45)


correlationMatrix = pd.DataFrame()
correlationMatrix["Smesh Sensor"] = splicedSmesh.to_list()
for i in range(len(dataArray)):

    splicedData = dataArray[i].loc[low:high]
    axes[i + 1].plot(splicedData.index,splicedData["pm2.5_atm"])
    print(len(splicedData["pm2.5_atm"]))
    correlationMatrix[purpleAirSensorLocations["Sensor Name"][i]] = splicedData["pm2.5_atm"].to_list()

    axes[i + 1].set_title(purpleAirSensorLocations["Sensor Name"][i])
    axes[i + 1].tick_params(axis='x', labelrotation=45)


# high = pd.Timestamp("2025-10-24-12")
# high = pepperwoodPreserve.index.max()

for ax in axes:

    ax.xaxis.set_major_formatter(mdates.DateFormatter('Oct %d %H:00',tz=tz.gettz("US/Pacific"))) 

plt.subplots_adjust(hspace=1.2)


plt.show()

In [ ]:
import seaborn as sns
corr = correlationMatrix.corr()

# Plot 5x5 correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("5x5 Correlation Matrix")
plt.show()